# The String Abstract Data Type: Sequences, Indexing, and Immutability

[Back to Session Overview](../README.md) · [Back to Module 01](../../README.md)

An **Abstract Data Type (ADT)** defines a mathematical model for data types where the type is defined by its behavior (semantics) from the point of view of a user of the data, specifically in terms of possible values and operations on data of this type.

In Python, text is modeled using the `str` class—an immutable sequence of Unicode code points. Under the hood (CPython PEP 393), strings are stored in optimized contiguous memory representations (Latin-1, UCS-2, or UCS-4 depending on the widest character present) with $O(1)$ random index access and $O(1)$ length queries.

### Learning Objectives
1. Understand the String Abstract Data Type and its contiguous memory layout.
2. Master forward (0-based) and reverse (negative) sequence indexing.
3. Apply string slicing arithmetic `[start:stop:step]` for extraction and reversal.
4. Parse and extract structured metadata from textual streams without external libraries.
5. Understand Python's immutability model, memory addressing (`id()`), and string interning.

In [ ]:
# String Creation, Length, and Forward/Backward Indexing
greeting: str = "Python Data Structures"

# Length query is O(1) in Python because CPython stores ob_size in the PyASCIIObject struct
length: int = len(greeting)
print(f"String: {greeting!r}, Length: {length}")

# Forward indexing starts at 0 and ends at length - 1
first_char: str = greeting[0]
second_char: str = greeting[1]

# Backward (negative) indexing: -1 represents the last character
last_char: str = greeting[-1]
penultimate_char: str = greeting[-2]

print(f"First: {first_char!r}, Second: {second_char!r}")
print(f"Last: {last_char!r}, Penultimate: {penultimate_char!r}")

# Traversal patterns: by character vs. by index
chars_loop = list(greeting[:6])
index_loop = [greeting[i] for i in range(6)]

assert chars_loop == index_loop == ["P", "y", "t", "h", "o", "n"]
assert greeting[0] == greeting[-len(greeting)]
print("Indexing assertions passed successfully.")

## Slicing Mechanics and Substring Extraction

String slicing uses the half-open interval notation `s[start:stop:step]`, which extracts elements at indices $i$ satisfying:

$$\text{index}(k) = \text{start} + k \cdot \text{step} \quad \text{for } 0 \le k < N$$

Where the upper bound `stop` is non-inclusive (the interval $[\text{start}, \text{stop})$). If omitted:
- `start` defaults to `0` (or `len(s) - 1` if `step < 0`).
- `stop` defaults to `len(s)` (or before index `0` if `step < 0`).
- `step` defaults to `1`.

A negative step reverses the direction of iteration, allowing idioms such as `s[::-1]` for string reversal.

In [ ]:
# Slicing demonstration
data: str = "ArtificialIntelligence"

# Prefix extraction [0:10]
prefix: str = data[:10]

# Suffix extraction [10:22]
suffix: str = data[10:]

# Substring with step
every_second_char: str = data[::2]

# Negative step reversal
reversed_data: str = data[::-1]

print(f"Prefix: {prefix}")
print(f"Suffix: {suffix}")
print(f"Step 2: {every_second_char}")
print(f"Reversed: {reversed_data}")

assert prefix == "Artificial"
assert suffix == "Intelligence"
assert prefix + suffix == data
assert reversed_data[0] == data[-1] and reversed_data[-1] == data[0]
print("Slicing assertions passed successfully.")

## Substring Search and Parsing: Practical Case Study

A common engineering task in data preprocessing is extracting structured tokens from semi-structured log lines. Methods available on `str`:
- `str.find(sub[, start[, end]])`: Returns the lowest index where `sub` is found, or `-1` if not found.
- `str.index(sub)`: Similar to `find()`, but raises `ValueError` when absent.
- `str.rfind(sub)`: Returns the highest index where `sub` is found.

In [ ]:
# Case Study: Parsing email logs without regular expressions
header_line: str = "From stephen.marquard@uct.ac.za Sat Jan  5 09:14:16 2008"

# Locate the delimiter symbols '@' and the subsequent space
at_index: int = header_line.find("@")
space_after_at: int = header_line.find(" ", at_index)

# Slice the domain host
host_domain: str = header_line[at_index + 1 : space_after_at]

# Locate sender name starting after 'From '
sender_start: int = len("From ")
sender_address: str = header_line[sender_start:space_after_at]
username: str = header_line[sender_start:at_index]

print(f"Sender Address: {sender_address}")
print(f"User:           {username}")
print(f"Host Domain:    {host_domain}")

assert host_domain == "uct.ac.za"
assert username == "stephen.marquard"
assert sender_address == "stephen.marquard@uct.ac.za"
print("Log parsing assertions passed successfully.")

## String Immutability, Memory Addresses, and Interning

Strings in Python are **immutable**. Once a `str` object is created in memory, its characters cannot be modified in place. Any operation that appears to modify a string (such as concatenation `+`, `.replace()`, or `.strip()`) produces a brand new `str` object at a new memory address.

Python tracks object identity via `id(obj)`, which in CPython returns the virtual memory address of the underlying `PyObject` structure.

In [ ]:
import sys

# Immutability in action
s1: str = "Machine"
addr_before: int = id(s1)

try:
    s1[0] = "m"  # type: ignore[index]
except TypeError as exc:
    print(f"Caught expected error: {exc}")

# Concatenation creates a new object
s1 = s1 + " Learning"
addr_after: int = id(s1)
print(f"Address before: {hex(addr_before)}")
print(f"Address after:  {hex(addr_after)}")
assert addr_before != addr_after

# String interning: Compile-time constant identifiers share identical memory addresses
str_a: str = sys.intern("neural_network")
str_b: str = sys.intern("neural_network")
print(f"str_a id: {id(str_a)}, str_b id: {id(str_b)}")
assert str_a is str_b  # Exact memory address equality
print("Immutability and interning assertions passed successfully.")

## Practical Lab Exercise: Robust Text Sanitizer and Header Parser

Build two robust functions:
1. `extract_log_metadata(record: str) -> dict[str, str]`: Extracts `sender`, `domain`, and `timestamp` from a standard mail log line.
2. `clean_and_tokenize(text: str) -> list[str]`: Removes punctuation, converts to lower case, and splits into whitespace-delimited tokens.

In [ ]:
import string


def extract_log_metadata(record: str) -> dict[str, str]:
    if not record.startswith("From "):
        raise ValueError("Invalid log header line: missing From prefix")
    parts = record.split()
    email = parts[1]
    user, domain = email.split("@")
    timestamp = " ".join(parts[4:7])
    return {
        "sender": email,
        "user": user,
        "domain": domain,
        "timestamp": timestamp,
    }


def clean_and_tokenize(text: str) -> list[str]:
    table = str.maketrans("", "", string.punctuation)
    cleaned = text.translate(table).lower()
    return cleaned.split()


# Verification tests
log_sample = "From alice@research.org Fri Oct 24 14:22:10 2026"
meta = extract_log_metadata(log_sample)
assert meta["user"] == "alice"
assert meta["domain"] == "research.org"
assert meta["timestamp"] == "Oct 24 14:22:10"

sentence = "Data Structures, Algorithms & Machine Learning: Essential Tools!"
tokens = clean_and_tokenize(sentence)
assert "structures" in tokens
assert "algorithms" in tokens
assert "essential" in tokens
print("All string lab exercises verified successfully!")